In [0]:
-- ============================================================
-- Project: Databricks SQL Retail Analytics Dashboard
-- File: 01_business_kpis.sql
-- Purpose: Core commercial KPIs from the Gold analytics model
-- ============================================================

USE SCHEMA retail_analytics_project;


-- ============================================================
-- 1. Revenue overview
-- ============================================================

SELECT
    ROUND(SUM(gross_sales_amount), 2) AS gross_sales,
    ROUND(SUM(cancellation_amount), 2) AS cancellation_amount,
    ROUND(SUM(net_sales_amount), 2) AS net_sales
FROM fact_sales;


-- ============================================================
-- 2. Complete business KPI summary
-- ============================================================

SELECT
    COUNT(*) AS total_commercial_lines,

    COUNT(DISTINCT invoice_no) AS total_commercial_invoices,

    COUNT(DISTINCT CASE
        WHEN transaction_type = 'SALE'
        THEN invoice_no
    END) AS total_sales_invoices,

    COUNT(DISTINCT CASE
        WHEN transaction_type = 'CANCELLATION'
        THEN invoice_no
    END) AS total_cancellation_invoices,

    COUNT(DISTINCT CASE
        WHEN customer_id <> 'UNKNOWN'
        THEN customer_id
    END) AS known_customers,

    ROUND(SUM(gross_sales_amount), 2) AS gross_sales,

    ROUND(SUM(cancellation_amount), 2) AS cancellation_amount,

    ROUND(SUM(net_sales_amount), 2) AS net_sales,

    SUM(sold_quantity) AS sold_units,

    SUM(cancelled_quantity) AS cancelled_units,

    SUM(net_quantity) AS net_units

FROM fact_sales;


-- ============================================================
-- 3. Average Order Value
-- Calculated only from SALE invoices
-- ============================================================

WITH sales_invoice_totals AS (
    SELECT
        invoice_no,
        SUM(gross_sales_amount) AS invoice_gross_sales
    FROM fact_sales
    WHERE transaction_type = 'SALE'
    GROUP BY invoice_no
)

SELECT
    COUNT(*) AS total_sales_invoices,

    ROUND(
        AVG(invoice_gross_sales),
        2
    ) AS average_order_value,

    ROUND(
        MIN(invoice_gross_sales),
        2
    ) AS minimum_order_value,

    ROUND(
        MAX(invoice_gross_sales),
        2
    ) AS maximum_order_value

FROM sales_invoice_totals;


-- ============================================================
-- 4. Cancellation value rate
-- ============================================================

SELECT
    ROUND(
        SUM(gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(cancellation_amount)
        / NULLIF(SUM(gross_sales_amount), 0)
        * 100.0,
        2
    ) AS cancellation_value_rate_pct

FROM fact_sales;


-- ============================================================
-- 5. Cancellation unit rate
-- ============================================================

SELECT
    SUM(sold_quantity) AS sold_units,

    SUM(cancelled_quantity) AS cancelled_units,

    SUM(net_quantity) AS net_units,

    ROUND(
        SUM(cancelled_quantity)
        / NULLIF(SUM(sold_quantity), 0)
        * 100.0,
        2
    ) AS cancellation_unit_rate_pct

FROM fact_sales;


-- ============================================================
-- 6. Cancellation line rate
-- ============================================================

SELECT
    SUM(
        CASE
            WHEN transaction_type = 'SALE'
            THEN 1
            ELSE 0
        END
    ) AS sale_lines,

    SUM(
        CASE
            WHEN transaction_type = 'CANCELLATION'
            THEN 1
            ELSE 0
        END
    ) AS cancellation_lines,

    COUNT(*) AS total_commercial_lines,

    ROUND(
        SUM(
            CASE
                WHEN transaction_type = 'CANCELLATION'
                THEN 1
                ELSE 0
            END
        )
        / NULLIF(COUNT(*), 0)
        * 100.0,
        2
    ) AS cancellation_line_rate_pct

FROM fact_sales;


-- ============================================================
-- 7. Commercial summary by transaction type
-- ============================================================

SELECT
    transaction_type,

    COUNT(*) AS total_lines,

    COUNT(DISTINCT invoice_no) AS total_invoices,

    SUM(sold_quantity) AS sold_units,

    SUM(cancelled_quantity) AS cancelled_units,

    SUM(net_quantity) AS net_units,

    ROUND(
        SUM(gross_sales_amount),
        2
    ) AS gross_sales,

    ROUND(
        SUM(cancellation_amount),
        2
    ) AS cancellation_amount,

    ROUND(
        SUM(net_sales_amount),
        2
    ) AS net_sales

FROM fact_sales

GROUP BY
    transaction_type

ORDER BY
    total_lines DESC;